In [1]:
print("hallow world")

hallow world


In [2]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from datetime import datetime, timedelta, time
import os
import os
import sys
import pandas as pd
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings("ignore")
import plotly.express as px

In [3]:
dataset_path = r'C:\Users\TPWODL\New folder_Content\AutonomousDataAnalystAgent\data\raw_path\x_data.xlsx'

In [4]:
new_df = pd.read_excel(r'C:\Users\TPWODL\New folder_Content\AutonomousDataAnalystAgent\data\raw_path\x_data.xlsx')

In [5]:
df = new_df.copy()

In [6]:
consumer_number = df[df['CONSUMER NUMBER'].astype(str).str.match(r'^\d{12}$')]

In [7]:
filter_df = consumer_number[
    ~consumer_number["TWEET/LINK"].astype(str).str.contains("DM", case=False, na=False)
]

In [8]:
filter_df['X_USER_ID'] = filter_df['TWEET/LINK'].str.extract(r'(?:twitter\.com|x\.com)/([^/]+)/')

In [9]:
filter_df['X_USER_ID'] = filter_df['X_USER_ID'].fillna('').apply(lambda x: '@' + x if x and not x.startswith('@') else x)


In [10]:
df['duplicate_case'] = df['REMARKS'].astype(str).str.findall(r'(?i)\b\d{3,5}\b')

In [11]:
df['duplicate_case'] = df['duplicate_case'].astype(str).str.replace(r'[\[\]]', '', regex=True)


In [12]:
df['duplicate_case'] = df['duplicate_case'].astype(str).str.strip("'")

In [13]:
df['duplicate_case'].isnull().value_counts()


duplicate_case
False    32873
Name: count, dtype: int64

In [14]:
df['duplicate_case'].value_counts()


duplicate_case
                         26295
8505                        27
18325', '6920               27
17794', '6287', '2018       24
8724                        20
                         ...  
32862                        1
1468                         1
600                          1
2023', '2024                 1
612                          1
Name: count, Length: 2923, dtype: int64

In [15]:
df['duplicate_case'].count()

np.int64(32873)

In [16]:
# Count how many rows are blank (NaN or empty string)
blank_count = df['duplicate_case'].astype(str).str.strip().replace('', pd.NA).isna().sum()

# Count how many rows are filled (not blank)
filled_count = df['duplicate_case'].astype(str).str.strip().replace('', pd.NA).notna().sum()

# Total rows
total_rows = len(df)

print("Total rows:", total_rows)
print("Filled rows:", filled_count)
print("Blank rows:", blank_count)


Total rows: 32873
Filled rows: 6578
Blank rows: 26295


In [17]:
df['appreciation_tweet'] = df['REMARKS'].astype(str).str.findall(r'(?i)\bappreciation\s*tweet\b')

In [18]:
df['appreciation_tweet'] = df['appreciation_tweet'].astype(str).str.replace(r'[\[\]]', ' ', regex=True)


In [19]:
df['appreciation_tweet'] = df['appreciation_tweet'].astype(str).str.replace("'", "", regex=False)


In [20]:
df['appreciation_tweet'].astype(str).str.title().value_counts()

appreciation_tweet
                       29429
Appreciation Tweet      3444
Name: count, dtype: int64

In [21]:
df.columns

Index(['SL.NO', 'DATE', 'SHIFT DUTY', 'QUERY/REQUEST/COMPLAINT',
       'COMPLAINT DETAILS', 'COMPLAINT NUMBER', 'SECTION', 'SUB-DIVISION',
       'DIVISION', 'CIRCLE', 'COMPLAINT TYPE', 'CONSUMER NUMBER',
       'MOBILE NUMB', 'DEPT', 'CLOSED/OPEN', 'REMARKS', 'TWEET/LINK',
       'COMPLAINANT NAME', 'COMPLAINT RECEIVED TIME', 'RESPONSE TIME',
       'SECOND RESPONSE TIME', 'FINAL RESPONSE TIME',
       'FINAL RESPONSE DATE DD/MM/YYYY', 'PSCC/FG/TO', 'ARREARS',
       'REPEAT (Y / N)', 'FORWARDED TO', 'SENTIMENTS', 'NATURE OF TWEET',
       'ACTIONABLE/ NON ACTIONABLE', 'AGEING', 'SLAB', 'SLAB2', 'MINUTE',
       'duplicate_case', 'appreciation_tweet'],
      dtype='object')

In [22]:
remark_df = df[['SL.NO', 'DATE',
                'COMPLAINT DETAILS',
                'DIVISION', 'CIRCLE', 'COMPLAINT TYPE', 'DEPT', 'CLOSED/OPEN',
                'TWEET/LINK','duplicate_case','appreciation_tweet']]


In [23]:
remark_df.columns

Index(['SL.NO', 'DATE', 'COMPLAINT DETAILS', 'DIVISION', 'CIRCLE',
       'COMPLAINT TYPE', 'DEPT', 'CLOSED/OPEN', 'TWEET/LINK', 'duplicate_case',
       'appreciation_tweet'],
      dtype='object')

In [24]:
remark_df.to_excel('airemk.xlsx')


In [25]:
remark_df.isnull().sum()


SL.NO                    0
DATE                     0
COMPLAINT DETAILS        0
DIVISION              7089
CIRCLE                7139
COMPLAINT TYPE           0
DEPT                     0
CLOSED/OPEN              0
TWEET/LINK               0
duplicate_case           0
appreciation_tweet       0
dtype: int64

In [26]:
selected_month = '2025-8'

In [27]:
def get_month_data(new_df, selected_month):
                """Filter data for selected month"""
                return df[df['DATE'].dt.to_period('M') == selected_month]

In [28]:
month_df = get_month_data(new_df, selected_month)

In [29]:
month_df.shape

(1303, 36)

In [30]:
def get_monthly_remarks_analysis(month_df):
                """Analyze REMARKS column for patterns"""
                temp_df = month_df.copy()
                
                if 'REMARKS' not in temp_df.columns:
                    return {
                        'Appreciation Tweets': 0,
                        'Awaited Consumer': 0,
                        '5-digit Numbers': 0,
                        'Total Remarks': 0
                    }
                
                temp_df['REMARKS'] = temp_df['REMARKS'].fillna('').astype(str)
                
                appreciation_count = temp_df['REMARKS'].str.contains("Appreciation Tweet", case=False, na=False).sum()
                awaited_consumer_count = temp_df['REMARKS'].str.contains("Awaited consumer", case=False, na=False).sum()
                number_count = temp_df['REMARKS'].str.contains(r"\b\d{5}\b", na=False).sum()

                return {
                    'Appreciation Tweets': int(appreciation_count),
                    'Awaited Consumer': int(awaited_consumer_count),
                    '5-digit Numbers': int(number_count),
                    'Total Remarks': len(temp_df)
                }

In [31]:
report = get_monthly_remarks_analysis(month_df)

In [32]:
dfers = pd.DataFrame([report])


In [33]:
dfers

,Appreciation Tweets,Awaited Consumer,5-digit Numbers,Total Remarks
0,154,231,303,1303


In [59]:
start_year="2025"
start_quarter="2" 
end_year="2025" 
end_quarter="2"

In [64]:

    
    # Convert DATE to quarterly period
new_df['QUARTER'] = new_df['DATE'].dt.to_period('Q')
    
    # Build start and end periods
start_period = pd.Period(f"{start_year}Q{start_quarter}", freq='Q')
end_period   = pd.Period(f"{end_year}Q{end_quarter}", freq='Q')

In [68]:
dfhh = new_df[(new_df['QUARTER'] >= start_period) & (new_df['QUARTER'] <= end_period)]


In [ ]:
dfhh = new_df[(new_df['QUARTER'] >= start_period) & (new_df['QUARTER'] <= end_period)]


In [60]:
import pandas as pd

def generate_quarter_wise_open_clode_pivot_report(dataset_path: str,
                                                  start_year: str,
                                                  start_quarter: str,
                                                  end_year: str,
                                                  end_quarter: str) -> dict:
    # Load dataset
    new_df = pd.read_excel(dataset_path)

    # Clean and format columns
    new_df['DATE'] = pd.to_datetime(new_df['DATE'])
    
    # Convert DATE to quarterly period
    new_df['QUARTER'] = new_df['DATE'].dt.to_period('Q')
    
    # Build start and end periods
    start_period = pd.Period(f"{start_year}Q{start_quarter}", freq='Q')
    end_period   = pd.Period(f"{end_year}Q{end_quarter}", freq='Q')
    
    # Filter rows within the range
    df = new_df[(new_df['QUARTER'] >= start_period) & (new_df['QUARTER'] <= end_period)]

    # Clean text columns
    df['COMPLAINT TYPE'] = df['COMPLAINT TYPE'].astype(str).str.strip().str.title()
    df['DEPT'] = df['DEPT'].astype(str).str.strip().str.title()
    df['CLOSED/OPEN'] = df['CLOSED/OPEN'].astype(str).str.strip().str.title()

    # Pivot table
    pivot = pd.pivot_table(
        df,
        values='DATE',
        index=['COMPLAINT TYPE'],          # keep this index
        columns=['DEPT','CLOSED/OPEN'],
        aggfunc='count',
        fill_value=0,
        margins=True,
        margins_name='Grand Total',
        observed=False
    )

    # Flatten MultiIndex columns into single strings
    pivot.columns = [f"{dept}_{status}" for dept, status in pivot.columns]

    # Convert pivot table to dictionary format, preserving index
    pivot_dict = pivot.to_dict()

    return pivot_dict


In [61]:
pvt = generate_quarter_wise_open_clode_pivot_report(dataset_path,
                                                  start_year,
                                                  start_quarter,
                                                  end_year,
                                                  end_quarter,)

In [62]:
poiuu = pd.DataFrame(pvt)

In [63]:
poiuu

,Commercial_Closed,O&M_Closed,O&M_Open,Other_Closed,Grand Total_
Billing,64,0,0,0,64
Civil Works,0,88,20,0,108
Electrification,0,69,0,0,69
Low Voltage,0,283,0,0,283
Metering,6,0,0,0,6
No Power Supply,0,1569,0,0,1569
Nsc,89,0,0,0,89
Others,0,7,0,614,621
Payment,34,0,0,0,34
Pole Shifting / Lt Sagging,0,105,18,0,123


In [44]:
def generate_finance_year_wise_open_clode_pivot_report(dataset_path: str, start_year: str, start_date: str, end_year: str, end_date: str) -> dict:
    # Load dataset
    new_df = pd.read_excel(dataset_path)

    # Clean and format columns  
    new_df['DATE'] = pd.to_datetime(new_df['DATE'])
    
    # Filter by date range
    start_dt = pd.to_datetime(f"{start_year}-{start_date}")
    end_dt = pd.to_datetime(f"{end_year}-{end_date}")
    df = new_df[(new_df['DATE'] >= start_dt) & (new_df['DATE'] <= end_dt)]
    
    df['COMPLAINT TYPE'] = df['COMPLAINT TYPE'].astype(str).str.strip().str.title()
    df['DEPT'] = df['DEPT'].astype(str).str.strip().str.title()
    df['CLOSED/OPEN'] = df['CLOSED/OPEN'].astype(str).str.strip().str.title()

    pivot = pd.pivot_table(
        df,
        values='DATE',
        index=['COMPLAINT TYPE'],          # keep this index
        columns=['DEPT','CLOSED/OPEN'],
        aggfunc='count',
        fill_value=0,
        margins=True,
        margins_name='Grand Total',
        observed=False
    )

    # Flatten MultiIndex columns into single strings
    pivot.columns = [f"{dept}_{status}" for dept, status in pivot.columns]

    # Convert pivot table to dictionary format, preserving index
    pivot_dict = pivot.to_dict()

    return pivot_dict

In [46]:
# Example 1: Financial year filter (April 2023 to March 2024)
result = generate_finance_year_wise_open_clode_pivot_report(
    dataset_path,
    start_year="2024",
    start_date="04-01",  # April 1st
    end_year="2025",
    end_date="03-31"     # March 31st
)

In [47]:
result

{'Commercial_Closed': {'Billing': 226,
  'Civil Works': 0,
  'Electrification': 0,
  'Low Voltage': 0,
  'Metering': 64,
  'No Power Supply': 0,
  'Nsc': 281,
  'Others': 0,
  'Payment': 56,
  'Pole Shifting / Lt Sagging': 0,
  'Portal Problem': 79,
  'Power Outage': 0,
  'Safety': 0,
  'Solar': 26,
  'Transformer Failure / Ncc': 0,
  'Grand Total': 732},
 'O&M_Closed': {'Billing': 0,
  'Civil Works': 571,
  'Electrification': 283,
  'Low Voltage': 470,
  'Metering': 0,
  'No Power Supply': 2273,
  'Nsc': 0,
  'Others': 35,
  'Payment': 0,
  'Pole Shifting / Lt Sagging': 396,
  'Portal Problem': 0,
  'Power Outage': 4142,
  'Safety': 580,
  'Solar': 0,
  'Transformer Failure / Ncc': 275,
  'Grand Total': 9025},
 'O&M_Open': {'Billing': 0,
  'Civil Works': 14,
  'Electrification': 0,
  'Low Voltage': 0,
  'Metering': 0,
  'No Power Supply': 0,
  'Nsc': 0,
  'Others': 0,
  'Payment': 0,
  'Pole Shifting / Lt Sagging': 13,
  'Portal Problem': 0,
  'Power Outage': 0,
  'Safety': 11,
  'Sol

In [ ]:
result